In [1]:
def read_fasta(path):
    d = {}
    with open(path) as f:
        for x in f.read().strip().split(">"):
            if x:
                name, _, seq = x.partition("\n")
                d[name] = "".join(seq.split())
    return d

In [2]:
seqs = read_fasta("/home/chenyh/workspace/Alright-strain/data/raw/nextclade/H1N1/origin_seqs/AA/deduplicated.fasta")

In [3]:
ha_seqs = {header: seq for header, seq in seqs.items() if "|HA|" in header}

In [4]:
len(ha_seqs)

56939

In [5]:
from datetime import date

def collection_date(header: str) -> date | None:
    """Return the collection date stored in field 7 of this FASTA header."""
    try:
        return date.fromisoformat(header.split('|')[6])
    except (IndexError, ValueError):
        return None

lengths = {k: len(v) for k, v in ha_seqs.items()}
print(f"长度筛选前范围: {min(lengths.values())}–{max(lengths.values())}")

ha_seqs = {header: seq for header, seq in ha_seqs.items() if len(seq) == 566}
print(f"长度为 566 的 HA 序列: {len(ha_seqs):,}")

cutoff = date(2017, 1, 1)
dated_ha_seqs = {header: (seq, collection_date(header)) for header, seq in ha_seqs.items()}
missing_date_count = sum(collected is None for _, collected in dated_ha_seqs.values())
ha_seqs = {
    header: seq for header, (seq, collected) in dated_ha_seqs.items()
    if collected is not None and collected > cutoff
}
print(f"缺失或无效采样日期而排除的序列: {missing_date_count:,}")
print(f"采样日期晚于 {cutoff.isoformat()} 的序列: {len(ha_seqs):,}")

长度筛选前范围: 2–582
长度为 566 的 HA 序列: 47,075
缺失或无效采样日期而排除的序列: 2
采样日期晚于 2017-01-01 的序列: 34,358


In [6]:
with open("./ha_unaglined.fasta", "w") as f:
    for header, seq in ha_seqs.items():
        f.write(f">{header}\n{seq}\n")